In [0]:
# ============================================
# 1. CONFIGURACIÓN Y LECTURA DE BRONZE
# ============================================

from pyspark.sql.functions import (
    col, trim, lower, upper, to_timestamp,
    when, count, lit, array, element_at,
    pmod, xxhash64, concat_ws
)

df_clientes = spark.table("workspace.olist_nuevo.bronze_clientes")
df_pedidos = spark.table("workspace.olist_nuevo.bronze_pedidos")
df_detalles = spark.table("workspace.olist_nuevo.bronze_detalles")
df_productos = spark.table("workspace.olist_nuevo.bronze_productos")
df_vendedores = spark.table("workspace.olist_nuevo.bronze_vendedores")
df_categorias = spark.table("workspace.olist_nuevo.bronze_categorias")

In [0]:
# ============================================
# GENERACIÓN DE NOMBRES BRASILEÑOS
# ============================================

nombres_brasilenos = [
    "Joao", "Maria", "Jose", "Ana", "Pedro",
    "Mariana", "Lucas", "Gabriel", "Julia", "Rafael",
    "Beatriz", "Matheus", "Camila", "Felipe", "Larissa",
    "Gustavo", "Amanda", "Bruno", "Isabela", "Thiago",
    "Fernanda", "Leonardo", "Carolina", "Eduardo", "Leticia",
    "Daniel", "Patricia", "Rodrigo", "Bianca", "Vinicius",
    "Juliana", "Henrique", "Luana", "Marcelo", "Sofia",
    "Andre", "Claudia", "Diego", "Valentina", "Ricardo"
]

apellidos_brasilenos = [
    "Silva", "Santos", "Oliveira", "Souza", "Pereira",
    "Costa", "Rodrigues", "Almeida", "Nascimento", "Lima",
    "Araujo", "Fernandes", "Carvalho", "Gomes", "Martins",
    "Rocha", "Ribeiro", "Alves", "Monteiro", "Mendes",
    "Barros", "Freitas", "Barbosa", "Pinto", "Moura",
    "Dias", "Castro", "Campos", "Cardoso", "Teixeira",
    "Moreira", "Vieira", "Andrade", "Nunes", "Marques",
    "Machado", "Batista", "Ramos", "Cavalcanti", "Correia"
]

array_nombres = array(*[lit(n) for n in nombres_brasilenos])
array_apellidos = array(*[lit(a) for a in apellidos_brasilenos])

# NOMBRES DE CLIENTES
df_nombres_clientes = (
    df_clientes
    .select("customer_unique_id")
    .filter(col("customer_unique_id").isNotNull())
    .dropDuplicates(["customer_unique_id"])
    .withColumn(
        "nombre_cliente",
        concat_ws(
            " ",
            element_at(
                array_nombres,
                (pmod(
                    xxhash64(col("customer_unique_id"), lit("nombre")),
                    lit(len(nombres_brasilenos))
                ) + 1).cast("int")
            ),
            element_at(
                array_apellidos,
                (pmod(
                    xxhash64(col("customer_unique_id"), lit("apellido")),
                    lit(len(apellidos_brasilenos))
                ) + 1).cast("int")
            )
        )
    )
)

# NOMBRES DE VENDEDORES
df_nombres_vendedores = (
    df_vendedores
    .select("seller_id")
    .filter(col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])
    .withColumn(
        "nombre_vendedor",
        concat_ws(
            " ",
            element_at(
                array_nombres,
                (pmod(
                    xxhash64(col("seller_id"), lit("nombre")),
                    lit(len(nombres_brasilenos))
                ) + 1).cast("int")
            ),
            element_at(
                array_apellidos,
                (pmod(
                    xxhash64(col("seller_id"), lit("apellido")),
                    lit(len(apellidos_brasilenos))
                ) + 1).cast("int")
            )
        )
    )
)

print("MUESTRA DE NOMBRES DE CLIENTES")
display(df_nombres_clientes.limit(10))

print("MUESTRA DE NOMBRES DE VENDEDORES")
display(df_nombres_vendedores.limit(10))

MUESTRA DE NOMBRES DE CLIENTES


customer_unique_id,nombre_cliente
861eff4711a542e4b93843c6dd7febb0,Matheus Cavalcanti
290c77bc529b7ac935b93aa66c333dc3,Gabriel Oliveira
060e732b5b29e8181a18229c7b0b2b5e,Julia Campos
259dac757896d24d7702b9acbbff3f3c,Leticia Marques
345ecd01c38d18a9036ed96c73b8d066,Rafael Ribeiro
4c93744516667ad3b8f1fb645a3116a4,Juliana Teixeira
addec96d2e059c80c30fe6871d30d177,Beatriz Campos
57b2a98a409812fe9618067b6b8ebe4f,Rodrigo Martins
1175e95fb47ddff9de6b2b06188f7e0d,Camila Castro
9afe194fb833f79e300e37e580171f22,Andre Rodrigues


MUESTRA DE NOMBRES DE VENDEDORES


seller_id,nombre_vendedor
3442f8959a84dea7ee197c632cb2df15,Patricia Moura
d1b65fc7debc3361ea86b5f14c68d2e2,Andre Cardoso
ce3ad9de960102d0677a81f5d0bb7b2d,Larissa Mendes
c0f3eea2e14555b6faeea3dd58c1b1c3,Isabela Dias
51a04a8a6bdcb23deccc82b0b80742cf,Valentina Moura
c240c4061717ac1806ae6ee72be3533b,Sofia Castro
e49c26c3edfa46d227d5121a6b6e4d37,Carolina Silva
1b938a7ec6ac5061a66a3766e0e75f90,Larissa Moura
768a86e36ad6aae3d03ee3c6433d61df,Julia Monteiro
ccc4bbb5f32a6ab2b7066a4130f114e3,Luana Andrade


In [0]:

# ============================================
# 1. LIMPIEZA DE CLIENTES
# ============================================

df_clientes_silver = (
    df_clientes
    .dropDuplicates(["customer_id"])
    .na.drop(subset=["customer_id", "customer_unique_id"])
    .filter(trim(col("customer_id")) != "")
    .filter(trim(col("customer_unique_id")) != "")
    .join(
        df_nombres_clientes,
        on="customer_unique_id",
        how="left"
    )
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .select(
        "customer_id",
        "customer_unique_id",
        "nombre_cliente",
        "customer_city",
        "customer_state"
    )
)


# ============================================
# 2. LIMPIEZA DE PEDIDOS ENTREGADOS
# ============================================

df_pedidos_silver = (
    df_pedidos
    .dropDuplicates(["order_id"])
    .na.drop(subset=["order_id", "customer_id", "order_status"])
    .filter(trim(col("order_id")) != "")
    .filter(trim(col("customer_id")) != "")
    .withColumn("order_status", lower(trim(col("order_status"))))
    .filter(col("order_status") == "delivered")
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp(col("order_purchase_timestamp"))
    )
    .filter(col("order_purchase_timestamp").isNotNull())
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    )
)


# ============================================
# 3. LIMPIEZA DE DETALLES
# SOLO PEDIDOS ENTREGADOS
# ============================================

df_detalles_silver = (
    df_detalles
    .dropDuplicates(["order_id", "order_item_id"])
    .na.drop(subset=[
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ])
    .filter(trim(col("order_id")) != "")
    .filter(trim(col("product_id")) != "")
    .filter(trim(col("seller_id")) != "")
    .filter(col("order_item_id") > 0)
    .filter(col("price") >= 0)
    .filter(col("freight_value") >= 0)
    .withColumn(
        "shipping_limit_date",
        to_timestamp(col("shipping_limit_date"))
    )
    .join(
        df_pedidos_silver.select("order_id"),
        on="order_id",
        how="left_semi"
    )
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    )
)


# ============================================
# 4. LIMPIEZA DE PRODUCTOS
# ============================================

df_productos_silver = (
    df_productos
    .dropDuplicates(["product_id"])
    .na.drop(subset=["product_id"])
    .filter(trim(col("product_id")) != "")
    .filter(
        col("product_weight_g").isNull() |
        (col("product_weight_g") >= 0)
    )
    .filter(
        col("product_length_cm").isNull() |
        (col("product_length_cm") >= 0)
    )
    .filter(
        col("product_height_cm").isNull() |
        (col("product_height_cm") >= 0)
    )
    .filter(
        col("product_width_cm").isNull() |
        (col("product_width_cm") >= 0)
    )
    .withColumn(
        "product_category_name",
        when(
            trim(col("product_category_name")) == "",
            lit(None).cast("string")
        ).otherwise(trim(col("product_category_name")))
    )
    .withColumnRenamed(
        "product_name_lenght",
        "longitud_nombre_producto"
    )
    .withColumnRenamed(
        "product_description_lenght",
        "longitud_descripcion_producto"
    )
)


# ============================================
# 5. LIMPIEZA DE VENDEDORES
# ============================================

df_vendedores_silver = (
    df_vendedores
    .dropDuplicates(["seller_id"])
    .na.drop(subset=["seller_id"])
    .filter(trim(col("seller_id")) != "")
    .join(
        df_nombres_vendedores,
        on="seller_id",
        how="left"
    )
    .withColumn("seller_city", lower(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .select(
        "seller_id",
        "nombre_vendedor",
        "seller_city",
        "seller_state"
    )
)


# ============================================
# 6. LIMPIEZA DE CATEGORÍAS
# ============================================

df_categorias_silver = (
    df_categorias
    .dropDuplicates(["product_category_name"])
    .na.drop(subset=[
        "product_category_name",
        "product_category_name_english"
    ])
    .filter(trim(col("product_category_name")) != "")
    .filter(trim(col("product_category_name_english")) != "")
    .withColumn(
        "product_category_name",
        trim(col("product_category_name"))
    )
    .withColumn(
        "product_category_name_english",
        lower(trim(col("product_category_name_english")))
    )
    .select(
        "product_category_name",
        "product_category_name_english"
    )
)


In [0]:
# ESTADÍSTICAS DE DETALLES

display(df_detalles_silver.describe())

# ESTADÍSTICAS DE PRODUCTOS

display(df_productos_silver.describe())

# DISTRIBUCIÓN DE ESTADOS DE PEDIDOS

display(
    df_pedidos_silver
    .groupBy("order_status")
    .count()
    .orderBy("count", ascending=False)
)

# DISTRIBUCIÓN DE CATEGORÍAS

display(
    df_categorias_silver
    .groupBy("product_category_name_english")
    .count()
    .orderBy("count", ascending=False)
)

summary,order_id,order_item_id,product_id,seller_id,price,freight_value
count,110197,110197,110197,110197,110197,110197
mean,null,1.1981814386961533,null,null,119.9805630824691,19.948597874715553
stddev,null,0.7066761819827142,null,null,182.2994464475326,15.698136099019955
min,00010242fe8c5a6d1ba2dd792cb16214,1,00066f42aeeb9f3007548bb9d3f33c38,0015a82c2db000af6aaaf3ae2ecb0532,0.85,0.0
max,fffe41c64501cc87c801fd61db3f6244,21,fffe9eeff12fcbd74a2f2b007dde0c58,ffff564a4f9085cd26170f4732393726,6735.0,409.68


summary,product_id,product_category_name,longitud_nombre_producto,longitud_descripcion_producto,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32341,32341,32341,32341,32949,32949,32949,32949
mean,null,null,48.47694876472589,771.4952846232337,2.1889861166939797,2276.4724877841513,30.81507784758263,16.937661234028347,23.196728277034204
stddev,null,null,10.245740725237228,635.1152246349542,1.73676563793155,4282.038730976914,16.914458054065967,13.637554061749512,12.079047453227835
min,00066f42aeeb9f3007548bb9d3f33c38,agro_industria_e_comercio,5,4,1,0,7,2,6
max,fffe9eeff12fcbd74a2f2b007dde0c58,utilidades_domesticas,76,3992,20,40425,105,105,118


order_status,count
delivered,96478


product_category_name_english,count
fashion_childrens_clothes,1
music,1
diapers_and_hygiene,1
industry_commerce_and_business,1
signaling_and_security,1
la_cuisine,1
arts_and_craftmanship,1
home_confort,1
books_technical,1
construction_tools_lights,1


In [0]:
print(
    "Order ID nulos:",
    df_pedidos_silver.filter(col("order_id").isNull()).count()
)

Order ID nulos: 0


In [0]:
# VERIFICAR DUPLICADOS EN CADA DATAFRAME

display(
    df_clientes_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

display(
    df_pedidos_silver
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

display(
    df_detalles_silver
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

display(
    df_productos_silver
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
)

display(
    df_vendedores_silver
    .groupBy("seller_id")
    .count()
    .filter(col("count") > 1)
)

display(
    df_categorias_silver
    .groupBy("product_category_name")
    .count()
    .filter(col("count") > 1)
)

customer_id,count


order_id,count


order_id,order_item_id,count


product_id,count


seller_id,count


product_category_name,count


In [0]:
# GUARDAR TABLAS SILVER

df_clientes_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_clientes"
)

df_pedidos_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_pedidos"
)

df_detalles_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_detalles"
)

df_productos_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_productos"
)

df_vendedores_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_vendedores"
)

df_categorias_silver.write.mode("overwrite").saveAsTable(
    "workspace.olist_nuevo.silver_categorias"
)

In [0]:
print("========== CONTEO FINAL SILVER ==========")

print("Clientes:", spark.table("workspace.olist_nuevo.silver_clientes").count())

print("Pedidos:", spark.table("workspace.olist_nuevo.silver_pedidos").count())

print("Detalles:", spark.table("workspace.olist_nuevo.silver_detalles").count())

print("Productos:", spark.table("workspace.olist_nuevo.silver_productos").count())

print("Vendedores:", spark.table("workspace.olist_nuevo.silver_vendedores").count())

print("Categorías:", spark.table("workspace.olist_nuevo.silver_categorias").count())

========== CONTEO FINAL SILVER ==========
Clientes: 99441
Pedidos: 96478
Detalles: 110197
Productos: 32951
Vendedores: 3096
Categorías: 71


In [0]:
df_vendedores_originales = (
    spark.table("workspace.olist_nuevo.silver_detalles")
    .join(
        spark.table("workspace.olist_nuevo.silver_pedidos"),
        on="order_id",
        how="inner"
    )
    .select("seller_id")
    .distinct()
)

print("Total de vendedores:", df_vendedores_originales.count())

Total de vendedores: 2970
